In [1]:
from pathlib import Path

import click
import torch
import torchaudio
from torch import nn
from torch.utils.data import DataLoader

from barking.cnn import CNNNetwork
from barking.dataset import UrbanSoundDataset
import barking.utils as util

In [6]:
annotations = "../data/UrbanSound8K/metadata/UrbanSound8K.csv"
audio_dir = "../data/UrbanSound8K/audio"
model = "../models/feedforwardnet.pth"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
click.echo("Using device: {}".format(device))
cnn = CNNNetwork().to(device)

state_dict = torch.load(model, map_location=device)
cnn.load_state_dict(state_dict)

mel_spect = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=1024,
    hop_length=512,
    n_mels=64,
).to(device)

learning_rate = 0.001
epochs = 100
batch_size = 128
class_mapping = [
    "air_conditioner",
    "car_horn",
    "children_playing",
    "dog_bark",
    "drilling",
    "engine_idling",
    "gun_shot",
    "jackhammer",
    "siren",
    "street_music",
]
sample_rate = 22050
num_samples = 22050

dataset = UrbanSoundDataset(
    annotations,
    audio_dir,
    device,
    mel_spect,
    22050,
    22050,
)

Using device: cuda:0


In [16]:
from random import randint

randidx = randint(0, len(dataset))
# get a sample from the dataset
sample, target = dataset[randidx][0], dataset[randidx][1]
# add extra dimension to input tensor
sample = sample.unsqueeze(0).to(device)

# make inference
predicted, expected = util.predict(cnn, sample, target, class_mapping)
# print the results
print(f"Predicted: '{predicted}', Expected: '{expected}'")

Predicted: 'siren', Expected: 'jackhammer'
